In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.output_parsers import PydanticToolsParser, StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.chat_models import ChatOllama
from langchain_core.runnables import RunnableLambda

In [2]:
#1. Prompt generation
ProjectIdea = ChatPromptTemplate.from_messages(
    [
        ("system", """You are a system planner for a multi-agent AI software team.

Divide the project into EXACTLY 5 sections:

1. Product Manager Agent → Define requirements
2. Architect Agent → Design system
3. Developer Agent → Write code
4. Tester Agent → Create test cases
5. Reviewer Agent → Review code

Return output in STRICT structured format:

Product_Manager:
...

Architect:
...

Developer:
...

Tester:
...

Reviewer:
...
"""),
        ("human", "{text}")
    ]
)

In [3]:
llm_ola =  ChatOllama(model="mistral", temperature=0)

/var/folders/m_/mx41vc1907s1rtts5k3gf3100000gn/T/ipykernel_60798/1086653502.py:1: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import ChatOllama``.
  llm_ola =  ChatOllama(model="mistral", temperature=0)


In [4]:
str_parser = StrOutputParser()

In [5]:

def dictionary_maker(text:str) -> dict:
    return {"text": text}

#Converting the dictionary_maker into a runnable lambda for chaining
dictionary_maker_runnable = RunnableLambda(dictionary_maker)

In [6]:
def Product_Manager(text):
    text = text["text"]
    PromptTemplate = ChatPromptTemplate.from_messages(
        [
            ("system", "you are the Product Manager of a company, search for the section that deals with product management and convert it into clear requirements"),
            ("human", "Give me the requirements, Features list, API endpoints and Constraints for the following project: {text}")
        ]
    )

    llm_ola =  ChatOllama(model="mistral", temperature=0)
    str_parser = StrOutputParser()
    chain = PromptTemplate | llm_ola | str_parser
    LLM_response = chain.invoke(text)
    return LLM_response

Product_Manager_agent = RunnableLambda(Product_Manager)    

In [7]:
def Architect (text):
    text = text["text"]
    PromptTemplate = ChatPromptTemplate.from_messages(
        [
            ("system", "you are the Architect of a company responsible for Architect and design system structure"),
            ("human", "Give me the design, Folder structure, Tech stack decisions, DB schema for the following project: {text}")
        ]
    )

    llm_ola =  ChatOllama(model="mistral", temperature=0)
    str_parser = StrOutputParser()
    chain = PromptTemplate | llm_ola | str_parser
    LLM_response = chain.invoke(text)
    return LLM_response

Architect_agent = RunnableLambda(Architect)   

In [8]:
def Developer (text):
    text = text["text"]
    PromptTemplate = ChatPromptTemplate.from_messages(
        [
            ("system", "you are the Developer of a company responsible for writting the code"),
            ("human", "Write the basic code for the following project: {text}")
        ]
    )

    llm_ola =  ChatOllama(model="mistral", temperature=0)
    str_parser = StrOutputParser()
    chain = PromptTemplate | llm_ola | str_parser
    LLM_response = chain.invoke(text)
    return LLM_response

Developer_gent = RunnableLambda(Developer)   

In [9]:
def Tester (text):
    text = text["text"]
    PromptTemplate = ChatPromptTemplate.from_messages(
        [
            ("system", "you are the Tester of a company responsible for testing"),
            ("human", "Generate test cases for the following: {text}")
        ]
    )

    llm_ola =  ChatOllama(model="mistral", temperature=0)
    str_parser = StrOutputParser()
    chain = PromptTemplate | llm_ola | str_parser
    LLM_response = chain.invoke(text)
    return LLM_response

Developer_gent = RunnableLambda(Tester)   

In [10]:

def Reviewer (text):
    text = text["text"]
    PromptTemplate = ChatPromptTemplate.from_messages(
        [
            ("system", "you are the Reviewer of a company Review the works"),
            ("human", "Review everything for the following : {text}")
        ]
    )

    llm_ola =  ChatOllama(model="mistral", temperature=0)
    str_parser = StrOutputParser()
    chain = PromptTemplate | llm_ola | str_parser
    LLM_response = chain.invoke(text)
    return LLM_response

Reviewer_agent = RunnableLambda(Reviewer)   

In [11]:
# from langchain_core.runnables import RunnableParallel, RunnableLambda
# chain = ProjectIdea | llm_ola | str_parser | dictionary_maker_runnable | RunnableParallel(branches = {"Product_Manager": Product_Manager, "Architect": Architect, "Developer": Developer, "Tester": Tester, "Reviewer": Reviewer})
# Response = chain.invoke({"text": "Build a REST API for a task manager app using Java Spring Boot"})
# print(Response)

In [12]:
def run_pipeline(user_input):
    # Step 1: (Optional) Planner
    plan = (ProjectIdea | llm_ola | str_parser).invoke({"text": user_input})

    # Step 2: Product Manager
    pm_output = Product_Manager({"text": plan})

    # Step 3: Architect
    arch_output = Architect({"text": pm_output})

    # Step 4: Developer
    dev_output = Developer({"text": arch_output})

    # Step 5: Tester
    test_output = Tester({"text": dev_output})

    # Step 6: Reviewer
    review_output = Reviewer({
        "text": f"Code:\n{dev_output}\n\nTests:\n{test_output}"
    })

    return {
        "Plan": plan,
        "PM": pm_output,
        "Architect": arch_output,
        "Developer": dev_output,
        "Tester": test_output,
        "Reviewer": review_output
    }


Response = run_pipeline("Build a REST API for a task manager app using Java Spring Boot")
print(Response)

{'Plan': ' Product_Manager:\n- Define the requirements for the REST API Task Manager App, including features such as user authentication, task creation, assignment, and completion.\n- Prioritize the features based on business needs and user feedback.\n- Collaborate with stakeholders to ensure that the requirements are clear and well-defined.\n\nArchitect:\n- Design the system architecture for the REST API Task Manager App using Java Spring Boot.\n- Define the database schema, including tables for users, tasks, and assignments.\n- Create a high-level design of the application, including the controllers, services, and repositories.\n- Ensure that the design is scalable, secure, and efficient.\n\nDeveloper:\n- Write code for the REST API Task Manager App using Java Spring Boot.\n- Implement the features defined by the product manager, following the system architecture designed by the architect.\n- Write unit tests to ensure that the code is functioning correctly.\n- Collaborate with other